In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
# os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_TRACING_V2"]="true"
# os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
groq_api_key=os.getenv("GROQ_API_KEY")
# groq_api_key

In [4]:
from langchain_groq import ChatGroq

In [5]:
model=ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, client=<groq.resources.chat.completions.Completions object at 0x000001EDCCE99EA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EDCCE99DB0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage
messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

response=model.invoke(messages)

In [12]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(response)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Source text: "Hello How are you?"\n   - Target language: French\n   - Task: Translation\n\n2.  **Identify Key Components:**\n   - "Hello" -> Greeting\n   - "How are you?" -> Question asking about well-being\n   - Note: There\'s a missing punctuation mark in the original ("Hello How are you?"), but it\'s clearly two parts: a greeting and a question.\n\n3.  **Determine French Equivalents:**\n   - "Hello" -> "Bonjour" (standard, polite) or "Salut" (informal)\n   - "How are you?" -> "Comment allez-vous ?" (formal/plural) or "Comment vas-tu ?" (informal singular) or "Ça va ?" (casual)\n   - Since no context is given, I\'ll provide the most common/standard translation, but I can also note formal vs. informal options if needed. However, for a direct translation, "Bonjour, comment allez-vous ?" or "Bonjour, comment vas-tu ?" are appropriate. I\'ll go with the standard neutral/polite form: "Bonjour, comment allez-vous 

In [13]:
# Using LCEL, we can chain the component
chain=model|parser
chain.invoke(messages)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input: "Hello How are you?"\n   - Task: Translate from English to French\n   - Note: There\'s a missing punctuation mark after "Hello" in the original, but it\'s clearly two phrases: "Hello" and "How are you?"\n\n2.  **Identify Key Components:**\n   - "Hello" -> Common French greetings: "Bonjour", "Salut" (informal)\n   - "How are you?" -> Common French phrases: "Comment allez-vous ?" (formal/plural), "Comment vas-tu ?" (informal singular), "Comment ça va ?" (neutral/casual)\n\n3.  **Determine Appropriate Translation:**\n   - Since no context is provided, I should provide a standard, polite translation that works in most situations.\n   - "Bonjour, comment allez-vous ?" (formal/standard)\n   - Alternatively, "Bonjour, comment ça va ?" (common/everyday)\n   - I\'ll go with the most standard and widely applicable: "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?"\n   - Actually, "Bonjour, comment all

In [14]:
# Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system", generic_template), ("user", "{text}")]
)

In [16]:
result=prompt.invoke({"language":"French", "text":"Hello"})

In [17]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [18]:
chain=prompt|model|parser

In [19]:
chain.invoke({"language":"French", "text":"Hello"})

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Translate the following into French:"\n   - Then provides: "Hello"\n   - Task: Translate "Hello" into French.\n\n2.  **Identify Key Translation:**\n   - "Hello" in English is a common greeting.\n   - In French, the most common and direct translations are:\n     - "Bonjour" (formal/general)\n     - "Salut" (informal)\n   - Since no context is provided, "Bonjour" is the standard, safest translation.\n\n3.  **Formulate Response:**\n   - Keep it concise as requested.\n   - Provide the direct translation: "Bonjour"\n   - Optionally note formality if needed, but usually just the word is fine.\n\n   Draft: "Bonjour"\n\n4.  **Verify:**\n   - Does "Bonjour" accurately translate "Hello" into French? Yes.\n   - Is it appropriate for general use? Yes.\n   - Format matches user request? Yes.\n\n   Final output: "Bonjour" (or "Bonjour !" with punctuation if preferred, but just the word is standard). I\'ll stick 